# Crypto Volatility Radar

A small experiment with TimesFM-3. Eventually this will forecast volatility
and volume for BTC, ETH, and SOL. For now I just want to prove the model can
handle the shape of the problem.

**Step 2:** run one synthetic multivariate forecast on a Colab GPU.

## Before running

In Colab, go to **Runtime > Change runtime type** and choose a **T4 GPU**.
Then use **Runtime > Run all**.

The install and first model download can take a few minutes. The weights are
downloaded to Colab's temporary cache, not this GitHub repo. No API key is
needed for the public checkpoint.

In [ ]:
%pip install -q "timesfm[torch]==3.0.1"

## Quick runtime check

A T4 is the intended Colab setup. The small smoke test can fall back to CPU,
which is handy when checking the notebook locally.

In [ ]:
import numpy as np
import torch
from timesfm3 import TimesFM3Forecaster

np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU fallback is fine for this small test. Use a T4 in Colab.")

## Make some fake hourly data

There are six targets: a volatility-like and volume-like series for each
asset. The values are made up, but they have daily seasonality and a fake
event bump. This lets us check the multivariate and covariate plumbing before
involving real market data.

In [ ]:
ASSETS = ("BTC", "ETH", "SOL")
TARGETS = ("realized_volatility", "volume")
MODEL_ID = "google/timesfm-3.0-pytorch"
CONTEXT_HOURS = 128
HORIZON_HOURS = 24

total_hours = CONTEXT_HOURS + HORIZON_HOURS
hours = np.arange(total_hours)
hour_of_day = hours % 24

daily_sin = np.sin(2 * np.pi * hour_of_day / 24).astype(np.float32)
daily_cos = np.cos(2 * np.pi * hour_of_day / 24).astype(np.float32)

# A pretend scheduled announcement, known both before and after the cutoff.
event_flag = np.zeros(total_hours, dtype=np.float32)
event_flag[72:76] = 1
event_flag[136:140] = 1

rng = np.random.default_rng(42)
target_rows = []
target_names = []

for asset_number, asset in enumerate(ASSETS, start=1):
    volatility = (
        0.015 * asset_number
        + 0.003 * daily_sin
        + 0.012 * event_flag
        + rng.normal(0, 0.001, total_hours)
    )
    volume = (1_000_000_000 / asset_number) * (
        1
        + 0.20 * daily_cos
        + 0.45 * event_flag
        + rng.normal(0, 0.04, total_hours)
    )

    target_rows.extend([
        np.clip(volatility[:CONTEXT_HOURS], 0, None),
        np.clip(volume[:CONTEXT_HOURS], 0, None),
    ])
    target_names.extend([f"{asset}_volatility", f"{asset}_volume"])

context = np.asarray(target_rows, dtype=np.float32)
known_future = np.asarray(
    [daily_sin, daily_cos, event_flag],
    dtype=np.float32,
)

print("Targets:", context.shape)
print("Known-future covariates:", known_future.shape)
print("Series:", ", ".join(target_names))

## Load TimesFM-3 and forecast 24 hours

One 2D context array means the six series are forecast together. The three
covariates include their known values through the forecast horizon.

In [ ]:
forecaster = TimesFM3Forecaster.from_pretrained(
    MODEL_ID,
    device=device,
    per_core_batch_size=1,
)

prediction = forecaster.predict(
    context=context,
    horizon=HORIZON_HOURS,
    past_future_covariates=known_future,
    return_quantiles=True,
    make_positive=True,
)

print("Point forecast:", prediction.forecast.shape)
print("Quantiles:", prediction.quantiles.shape)

## Check the result

TimesFM-3 returns the median forecast plus nine quantiles (p10 through p90).
These checks are intentionally boring: right shape, real numbers, ordered
intervals, and the point forecast matching p50.

In [ ]:
expected_forecast_shape = (len(target_names), HORIZON_HOURS)
expected_quantile_shape = (len(target_names), HORIZON_HOURS, 9)

assert prediction.forecast.shape == expected_forecast_shape
assert prediction.quantiles.shape == expected_quantile_shape
assert np.isfinite(prediction.forecast).all()
assert np.isfinite(prediction.quantiles).all()
assert (np.diff(prediction.quantiles, axis=-1) >= 0).all()
np.testing.assert_allclose(
    prediction.forecast,
    prediction.quantiles[..., 4],
    rtol=1e-5,
    atol=1e-6,
)

print(f"Step 2 passed on {device}: TimesFM-3 ran a 6-series, 24-hour forecast.")
print("Forecast shape:", prediction.forecast.shape)
print("Quantile shape:", prediction.quantiles.shape)
print("Next: stop here. Real hourly market data comes in step 3.")

## Notes

This only proves that the model and data shapes work. Synthetic accuracy says
nothing about whether the crypto forecasts will be useful.

The TimesFM-3 weights currently use Google's non-commercial license, so this
repo is a learning project rather than a production trading system.

Sources checked September 4, 2026:
[official TimesFM repo](https://github.com/google-research/timesfm) ·
[model card](https://huggingface.co/google/timesfm-3.0-pytorch)

**Stop here after the final check passes.**